In [ ]:
# INSTALL DEPENDENCIES

import subprocess
subprocess.run(["pip", "install", "openai", "pandas", "tqdm"], check=True)

In [ ]:
# IMPORTS and CONFIGURATION

import os
import re
import csv
import json
import time
import logging
import sqlite3
import traceback
from pathlib import Path
from typing import List, Optional
from dataclasses import dataclass

import numpy as np
import pandas as pd
from openai import OpenAI

# PATHS
# Edit to match the directory layout. DATA_ROOT should be the folder containing one subfolder per book.
# Each book subfolder must contain:
#   <book_slug>_clean.txt              — chapter-level summary text (partitioned)
#   <book_slug>_characters_clean.txt   — curated character list

DATA_ROOT   = Path("./data/corpus")          # change to corpus path
OUTPUT_ROOT = Path("./data/networks")        # where combined output CSV is saved
PROGRESS_DB = Path("./data/extraction_progress.db")
LOG_FILE    = Path("./data/extraction.log")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# LOGGING 
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE, encoding="utf-8"),
        logging.StreamHandler(),
    ],
)
log = logging.getLogger(__name__)

# API CONFIGURATION
# Keys as environment variables. Set API keys before running:
#   On Mac/Linux (terminal):
#     export ACADEMICCLOUD_KEY_1="your_key_here"
#     export DEEPINFRA_KEY_1="your_key_here"

#   In a .env file (with python-dotenv):
#     pip install python-dotenv
#     from dotenv import load_dotenv; load_dotenv()

# Add or remove entries below to match the number of keys available. Any entry with an empty or missing key is automatically skipped.

# Read an API key from environment. Returns None if not set.
def _key(env_var: str) -> Optional[str]:
    val = os.environ.get(env_var, "").strip()
    return val if val else None

API_CONFIGS = [cfg for cfg in [
    # AcademicCloud (llama-3.3-70b-instruct)
    {
        "name":     "academiccloud_1",
        "api_key":  _key("ACADEMICCLOUD_KEY_1"),
        "base_url": "https://chat-ai.academiccloud.de/v1",
        "model":    "llama-3.3-70b-instruct",
    },
    {
        "name":     "academiccloud_2",
        "api_key":  _key("ACADEMICCLOUD_KEY_2"),
        "base_url": "https://chat-ai.academiccloud.de/v1",
        "model":    "llama-3.3-70b-instruct",
    },
    # DeepInfra (meta-llama/Llama-3.3-70B-Instruct-Turbo)
    {
        "name":     "deepinfra_1",
        "api_key":  _key("DEEPINFRA_KEY_1"),
        "base_url": "https://api.deepinfra.com/v1/openai",
        "model":    "meta-llama/Llama-3.3-70B-Instruct-Turbo",
    },
    {
        "name":     "deepinfra_2",
        "api_key":  _key("DEEPINFRA_KEY_2"),
        "base_url": "https://api.deepinfra.com/v1/openai",
        "model":    "meta-llama/Llama-3.3-70B-Instruct-Turbo",
    },
] if cfg["api_key"]]   # automatically drops entries without a key

# RATE LIMITS (per key, conservative) 
MAX_REQUESTS_PER_MINUTE = 13  # conservative 
MAX_RETRIES             = 3
RETRY_DELAY             = 5   # seconds between retry attempts

if not API_CONFIGS:
    raise EnvironmentError(
        "No API keys found. Set at least one of: "
        "ACADEMICCLOUD_KEY_1, DEEPINFRA_KEY_1, ... as environment variables."
    )

log.info(f"Corpus root : {DATA_ROOT}")
log.info(f"Output root : {OUTPUT_ROOT}")
log.info(f"Progress DB : {PROGRESS_DB}")
log.info(f"API keys loaded: {len(API_CONFIGS)} ({[c['name'] for c in API_CONFIGS]})")
print(f" Configuration loaded — {len(API_CONFIGS)} API key(s) active")
print(f" Effective throughput: ~{len(API_CONFIGS) * MAX_REQUESTS_PER_MINUTE} calls/min")

In [ ]:
# DATA STRUCTURES

@dataclass
class Partition:
    number:  int
    title:   str
    content: str

@dataclass
class Relationship:
    chapter:      int
    char_a:       str
    char_b:       str
    relationship: str   # positive | negative | neutral

print(" Data structures defined")

In [ ]:
# PROGRESS TRACKER

# SQLite-backed tracker: every completed partition is written to disk immediately. If the laptop crashes or wifi drops, no work is lost, the next run picks up from where it left off.
class ProgressTracker:
    def __init__(self, db_path: Path):
        self.db_path = db_path
        self._init_db()

    def _conn(self):
        return sqlite3.connect(self.db_path)

    def _init_db(self):
        with self._conn() as conn:
            conn.executescript('''
                CREATE TABLE IF NOT EXISTS book_status (
                    book_name   TEXT PRIMARY KEY,
                    status      TEXT,
                    total_parts INTEGER DEFAULT 0,
                    done_parts  INTEGER DEFAULT 0,
                    start_time  REAL,
                    updated     REAL,
                    error_msg   TEXT
                );
                CREATE TABLE IF NOT EXISTS partition_status (
                    book_name   TEXT,
                    part_num    INTEGER,
                    status      TEXT,
                    n_rels      INTEGER DEFAULT 0,
                    timestamp   REAL,
                    error_msg   TEXT,
                    PRIMARY KEY (book_name, part_num)
                );
                CREATE TABLE IF NOT EXISTS llm_usage (
                    id                INTEGER PRIMARY KEY AUTOINCREMENT,
                    book_name         TEXT,
                    part_num          INTEGER,
                    timestamp         REAL,
                    api_key           TEXT,
                    success           INTEGER,
                    proc_time         REAL,
                    prompt_tokens     INTEGER DEFAULT 0,
                    completion_tokens INTEGER DEFAULT 0
                );
            ''')

    def book_status(self, book_name: str) -> Optional[str]:
        with self._conn() as conn:
            row = conn.execute("SELECT status FROM book_status WHERE book_name=?", (book_name,)).fetchone()
        return row[0] if row else None

    def set_book_status(self, book_name: str, status: str, total_parts: int = 0, error_msg: str = None):
        now = time.time()
        with self._conn() as conn:
            conn.execute('''
                INSERT INTO book_status (book_name, status, total_parts, done_parts,
                                         start_time, updated, error_msg)
                VALUES (?,?,?,0,?,?,?)
                ON CONFLICT(book_name) DO UPDATE SET
                    status=excluded.status,
                    total_parts=excluded.total_parts,
                    updated=excluded.updated,
                    error_msg=excluded.error_msg
            ''', (book_name, status, total_parts, now, now, error_msg))

    def completed_partitions(self, book_name: str) -> List[int]:
        with self._conn() as conn:
            rows = conn.execute("SELECT part_num FROM partition_status WHERE book_name=? AND status='completed'", (book_name,)).fetchall()
        return [r[0] for r in rows]

    def mark_partition_done(self, book_name: str, part_num: int, n_rels: int):
        now = time.time()
        with self._conn() as conn:
            conn.execute('''
                INSERT INTO partition_status (book_name, part_num, status, n_rels, timestamp)
                VALUES (?,?,?,?,?)
                ON CONFLICT(book_name, part_num) DO UPDATE SET
                    status='completed', n_rels=excluded.n_rels, timestamp=excluded.timestamp
            ''', (book_name, part_num, 'completed', n_rels, now))
            conn.execute('''
                UPDATE book_status SET done_parts=done_parts+1, updated=?
                WHERE book_name=?
            ''', (now, book_name))

    def mark_partition_failed(self, book_name: str, part_num: int, error: str):
        now = time.time()
        with self._conn() as conn:
            conn.execute('''
                INSERT INTO partition_status (book_name, part_num, status, timestamp, error_msg)
                VALUES (?,?,'failed',?,?)
                ON CONFLICT(book_name, part_num) DO UPDATE SET
                    status='failed', timestamp=excluded.timestamp, error_msg=excluded.error_msg
            ''', (book_name, part_num, now, error))

    def log_llm_call(self, book_name, part_num, api_key, success,
                     proc_time, prompt_tokens=0, completion_tokens=0):
        with self._conn() as conn:
            conn.execute(
                "INSERT INTO llm_usage "
                "(book_name, part_num, timestamp, api_key, success, proc_time, "
                "prompt_tokens, completion_tokens) VALUES (?,?,?,?,?,?,?,?)",
                (book_name, part_num, time.time(), api_key,
                 int(success), proc_time, prompt_tokens, completion_tokens)
            )

    def summary(self):
        with self._conn() as conn:
            books   = conn.execute("SELECT status, COUNT(*) FROM book_status GROUP BY status").fetchall()
            calls   = conn.execute("SELECT COUNT(*), SUM(success) FROM llm_usage").fetchone()
        return {
            "books":         dict(books),
            "total_calls":   calls[0] or 0,
            "success_calls": calls[1] or 0,
        }

progress_tracker = ProgressTracker(PROGRESS_DB)
print(" Progress tracker initialised")
print(f" DB: {PROGRESS_DB}")

In [ ]:
# BOOK DIRECTORY DISCOVERY

# Scans root for book subdirectories that contain BOTH a non-empty *_clean.txt (chapter summary) and a *_characters_clean.txt (cast list). 
# Returns a list of (book_dir, slug, summary_file, characters_file) tuples.
def find_book_directories(root: Path = DATA_ROOT) -> list:
    results = []
    for book_dir in sorted(root.iterdir()):
        if not book_dir.is_dir():
            continue

        summary_files = [
            f for f in book_dir.glob("*_clean.txt")
            if "_characters_clean" not in f.name
        ]
        char_files = list(book_dir.glob("*_characters_clean.txt"))

        if not summary_files or not char_files:
            continue

        summary_file = summary_files[0]
        char_file    = char_files[0]

        # Skip books whose text files are empty (not yet filled in)
        if summary_file.stat().st_size == 0 or char_file.stat().st_size == 0:
            continue

        slug = book_dir.name   # used as the unique book identifier
        results.append((book_dir, slug, summary_file, char_file))

    return results

books = find_book_directories()
print(f" Found {len(books)} ready book directories under {DATA_ROOT}")
for book_dir, slug, *_ in books[:5]:
    print(f"  {slug}")
print("  ...")

In [ ]:
# PARTITION PARSER

PARTITION_RE = re.compile(
    r'===\s*PARTITION\s+(\d+)\s*:\s*(.+?)\s*===',
    re.IGNORECASE
)

# Splits _clean.txt on === PARTITION N: [title] === delimiters. Returns list of Partition objects.
def parse_partitions(clean_txt: str) -> List[Partition]:
    splits = list(PARTITION_RE.finditer(clean_txt))
    if not splits:
        log.warning("No partition delimiters found — check file format")
        return []

    parts = []
    for i, m in enumerate(splits):
        num     = int(m.group(1))
        title   = m.group(2).strip()
        start   = m.end()
        end     = splits[i+1].start() if i+1 < len(splits) else len(clean_txt)
        content = clean_txt[start:end].strip()
        if content:
            parts.append(Partition(number=num, title=title, content=content))
    return parts

# Returns character list as-is
def parse_characters(chars_txt: str) -> str:
    return chars_txt.strip()

print(" Parsers defined")

In [ ]:
#  API Client with Rate Limiting

class RateLimitedClient:

    # Single API key wrapper with per-minute rate limiting
    def __init__(self, name: str, api_key: str, base_url: str, model: str):
        self.name          = name
        self.model_name    = model 
        self.client        = OpenAI(api_key=api_key, base_url=base_url)
        self.call_times    = []   # timestamps of recent calls
        self.total_calls   = 0

    # Enforce MAX_REQUESTS_PER_MINUTE
    def _wait_if_needed(self):
        now = time.time()
        # Remove calls older than 60 seconds
        self.call_times = [t for t in self.call_times if now - t < 60]
        if len(self.call_times) >= MAX_REQUESTS_PER_MINUTE:
            wait = 60 - (now - self.call_times[0]) + 0.5
            log.info(f"  [{self.name}] Rate limit — waiting {wait:.1f}s")
            time.sleep(wait)

    # Returns (content, proc_time, prompt_tokens, completion_tokens)
    def call(self, messages: list, max_tokens: int = 1500) -> tuple:
        self._wait_if_needed()
        t0 = time.time()
        response = self.client.chat.completions.create(model=self.model_name, messages=messages, max_tokens=max_tokens, temperature=0.1)
        self.call_times.append(time.time())
        self.total_calls += 1

        content          = response.choices[0].message.content.strip()
        prompt_tokens    = response.usage.prompt_tokens     if response.usage else 0
        completion_tokens = response.usage.completion_tokens if response.usage else 0

        return content, time.time() - t0, prompt_tokens, completion_tokens

# Rotates across multiple API keys round-robin. Each key has its own rate limiter. Combined throughput = N_keys × 13 calls/min.
class MultiKeyCoordinator:
    def __init__(self, configs: list):
        self.clients = [
            RateLimitedClient(c["name"], c["api_key"], c["base_url"], c["model"])
            for c in configs
        ]
        self._idx = 0

    # Returns (content, api_key_name, proc_time, prompt_tokens, completion_tokens)
    def call(self, messages: list) -> tuple:
        client = self.clients[self._idx % len(self.clients)]
        self._idx += 1
        content, t, pt, ct = client.call(messages)
        return content, client.name, t, pt, ct


coordinator = MultiKeyCoordinator(API_CONFIGS)
print(f" Multi-key coordinator ready ({len(API_CONFIGS)} key(s))")
print(f" Effective throughput: ~{len(API_CONFIGS) * MAX_REQUESTS_PER_MINUTE}/min")

In [ ]:
# LLM PROMPT 

def build_prompt(book_name: str, character_context: str,
                 partition: Partition, history_context: str) -> str:
    return f"""You are analyzing character relationships in "{book_name}" with a focus on PRECISION and CONTEXT.

CHARACTERS IN THIS STORY:
{character_context}

CURRENT PARTITION: {partition.title} (Chapter {partition.number})
PARTITION CONTENT:
{partition.content}
{history_context}

TASK: Extract character relationships considering:
1. Direct interactions in THIS partition
2. Relationship changes from previous chapters
3. Both explicit and implicit relationship indicators

RELATIONSHIP TYPES:
- POSITIVE: cooperation, friendship, support, alliance, shared goals
- NEGATIVE: conflict, hostility, opposition, betrayal, competing interests  
- NEUTRAL: professional interaction, acquaintance, unclear sentiment

IMPORTANT RULES:
1. Include relationships even if they don't change from previous chapters
2. Consider subtle relationship indicators (tone, cooperation level)
3. Use canonical character names from the character list
4. If uncertain about relationship type, choose neutral
5. Include relationships for major characters even if interaction is brief

OUTPUT FORMAT: Return ONLY a JSON array of relationships:
[
  {{
    "chapter": {partition.number},
    "char_a": "Character Name",
    "char_b": "Character Name", 
    "relationship": "positive|negative|neutral"
  }}
]

If no clear relationships are found, return []."""

def build_history_context(all_rels: list, current_num: int, max_recent: int = 3) -> str:
    if not all_rels:
        return ""
    recent = [r for r in all_rels if r["chapter"] >= current_num - max_recent]
    if not recent:
        return ""
    lines = ["\nPREVIOUS RELATIONSHIP CONTEXT (recent partitions):"]
    for r in recent[-30:]:
        lines.append(f"  Ch{r['chapter']}: {r['char_a']} ↔ {r['char_b']} [{r['relationship']}]")
    return "\n".join(lines)

print(" Prompt builder defined")

In [ ]:
# LLM CALL WITH RETRY

# Returns (list_of_rel_dicts, api_key_name). Retries up to MAX_RETRIES times on JSON parse failure or HTTP error
def call_llm_with_retry(book_name: str, character_context: str, partition: Partition, history_context: str) -> tuple:
    prompt = build_prompt(book_name, character_context, partition, history_context)
    messages = [{"role": "user", "content": prompt}]

    for attempt in range(MAX_RETRIES):
        try:
            # raw, api_name, proc_time = coordinator.call(messages)
            raw, api_name, proc_time, prompt_tokens, completion_tokens = coordinator.call(messages)

            # Strip markdown code fences
            raw = re.sub(r'^```(?:json)?\s*', '', raw)
            raw = re.sub(r'\s*```$', '', raw).strip()

            rels = json.loads(raw)
            if isinstance(rels, list):
                # Validate each entry
                valid = []
                for r in rels:
                    if all(k in r for k in ["chapter","char_a","char_b","relationship"]):
                        if r["relationship"] in ("positive","negative","neutral"):
                            valid.append(r)
                progress_tracker.log_llm_call(
                    book_name, partition.number, api_name, True, proc_time, prompt_tokens, completion_tokens)
                return valid, api_name

        except json.JSONDecodeError as e:
            log.warning(f"  JSON error attempt {attempt+1}/{MAX_RETRIES}: {e}")
        except Exception as e:
            log.error(f"  Error attempt {attempt+1}/{MAX_RETRIES}: {e}")
            if attempt < MAX_RETRIES - 1:
                time.sleep(RETRY_DELAY * (attempt + 1))

    progress_tracker.log_llm_call(book_name, partition.number, "unknown", False, 0)
    return [], "unknown"

print(" LLM caller with retry defined")

In [ ]:
# PER-BOOK PROCESSOR

# Processes one book. Writes results to:
#   book_dir/<base_name>_network.csv        ← main output
#   book_dir/<base_name>_processing_log.csv ← per-partition audit trail

# Returns True on success, False on failure. Skips partitions already completed (crash-safe resume)
def process_book(book_dir: Path, base_name: str, clean_file: Path, chars_file: Path) -> bool:
    book_status = progress_tracker.book_status(base_name)
    if book_status == "completed":
        log.info(f"SKIP (already done): {base_name}")
        return True

    log.info(f"\n{'='*60}")
    log.info(f"PROCESSING: {base_name}")

    try:
        clean_txt = clean_file.read_text(encoding="utf-8").strip()
        chars_txt = chars_file.read_text(encoding="utf-8").strip()
    except Exception as e:
        log.error(f"  Cannot read files: {e}")
        progress_tracker.set_book_status(base_name, "failed", error_msg=str(e))
        return False

    partitions = parse_partitions(clean_txt)
    if not partitions:
        log.warning(f"  No partitions parsed — check === PARTITION N: === format")
        progress_tracker.set_book_status(base_name, "failed", error_msg="no partitions found")
        return False

    character_context = parse_characters(chars_txt)
    log.info(f"  Partitions: {len(partitions)}")

    # Book name for LLM prompt: clean up the folder slug
    book_name = base_name.replace("-", " ").replace("_", " ")

    progress_tracker.set_book_status(base_name, "processing", total_parts=len(partitions))

    # Load already-completed partitions (resume support) 
    done_parts = set(progress_tracker.completed_partitions(base_name))

    # Load any existing network rows (for history context)
    network_file = book_dir / f"{base_name}_network.csv"
    all_rels = []
    if network_file.exists() and network_file.stat().st_size > 0:
        try:
            existing = pd.read_csv(network_file)
            all_rels = existing.to_dict("records")
            log.info(f"  Resuming: {len(done_parts)} partitions already done")
        except Exception:
            pass

    # Per-partition log (append mode — survives crashes)
    log_file = book_dir / f"{base_name}_processing_log.csv"
    log_fields = ["partition_num","partition_title","n_relationships",
                  "api_key","timestamp","status"]
    if not log_file.exists():
        with open(log_file, "w", newline="", encoding="utf-8") as f:
            csv.DictWriter(f, fieldnames=log_fields).writeheader()

    # Process each partition 
    for partition in partitions:
        if partition.number in done_parts:
            log.info(f"  Partition {partition.number}: SKIP (done)")
            continue

        log.info(f"  Partition {partition.number}: {partition.title[:50]}")
        history_ctx = build_history_context(all_rels, partition.number)

        rels, api_name = call_llm_with_retry(
            book_name, character_context, partition, history_ctx)

        all_rels.extend(rels)

        # Write network CSV after EVERY partition (crash-safe)
        if all_rels:
            # pd.DataFrame(all_rels).to_csv(network_file, index=False, encoding="utf-8")
            df_out = pd.DataFrame(all_rels).rename(columns={
                "chapter":      "Chapter",
                "char_a":       "Character A",
                "char_b":       "Character B",
                "relationship": "Relationship",
            })
            df_out.to_csv(network_file, index=False, encoding="utf-8")

        # Append to per-partition log 
        with open(log_file, "a", newline="", encoding="utf-8") as f:
            csv.DictWriter(f, fieldnames=log_fields).writerow({
                "partition_num":   partition.number,
                "partition_title": partition.title,
                "n_relationships": len(rels),
                "api_key":         api_name,
                "timestamp":       time.strftime("%Y-%m-%d %H:%M:%S"),
                "status":          "completed" if rels is not None else "failed",
            })

        # Update progress DB
        progress_tracker.mark_partition_done(base_name, partition.number, len(rels))
        log.info(f"    → {len(rels)} relationships extracted")

    # Mark book complete
    progress_tracker.set_book_status(base_name, "completed")
    log.info(f"   DONE: {base_name} | {len(all_rels)} total relationships")
    return True

print(" Per-book processor defined")

OPTION A: Sequential execution (simpler, one book at a time)
Use this if you have only 1–2 API keys, or for debugging. For faster extraction with multiple keys, use the next cell (threaded).

In [ ]:
# MAIN LOOP

def run_all():
    books = find_book_directories()
    log.info(f"\n{'='*60}")
    log.info(f"STARTING EXTRACTION")
    log.info(f"Books to process: {len(books)}")
    log.info(f"API keys: {len(API_CONFIGS)}")
    log.info(f"{'='*60}\n")

    t0 = time.time()
    ok, fail, skip = 0, 0, 0

    for book_dir, base_name, clean_file, chars_file in books:
        status = progress_tracker.book_status(base_name)
        if status == "completed":
            skip += 1
            continue
        success = process_book(book_dir, base_name, clean_file, chars_file)
        if success:
            ok += 1
        else:
            fail += 1

    elapsed = time.time() - t0
    log.info(f"\n{'='*60}")
    log.info(f"EXTRACTION COMPLETE")
    log.info(f"  OK:      {ok}")
    log.info(f"  Failed:  {fail}")
    log.info(f"  Skipped: {skip}")
    log.info(f"  Time:    {elapsed/60:.1f} min")

    s = progress_tracker.summary()
    log.info(f"  Total LLM calls: {s['total_calls']} ({s['success_calls']} successful)")

# Run a single book test first. Uncomment to test on one book before running all:
books = find_book_directories()
# process_book(*books[0])   # processes first book found

# Run everything
run_all()

OPTION B: Threaded execution (one thread per API key)
Recommended when you have 3+ API keys. Each thread owns one API key; books are distributed via a shared queue. 

In [ ]:
# THREADED MAIN LOOP (replaces sequential run_all)

import threading
from queue import Queue, Empty
import sqlite3

# Enable WAL mode so multiple threads can read/write SQLite concurrently
with sqlite3.connect(PROGRESS_DB) as conn:
    conn.execute("PRAGMA journal_mode=WAL")
print(" SQLite WAL mode enabled")

# Each worker thread owns exactly one API client. Pulls books from the shared queue until empty.
def worker(thread_id: int, client_config: dict, book_queue: Queue,
           results: dict, lock: threading.Lock):
    
    # Create a single-key coordinator for this thread
    single_coordinator = MultiKeyCoordinator([client_config])

    while True:
        try:
            book_dir, base_name, clean_file, chars_file = book_queue.get(timeout=3)
        except Empty:
            break

        # Check if already done (another thread may have grabbed it)
        status = progress_tracker.book_status(base_name)
        if status == "completed":
            book_queue.task_done()
            continue

        log.info(f"[Thread-{thread_id}] START: {base_name}")

        # Temporarily swap coordinator to this thread's single-key version. We do this by patching locally — safer than shared state
        global coordinator
        original = coordinator

        # Thread-local coordinator swap using lock
        try:
            success = _process_book_with_client(book_dir, base_name, clean_file, chars_file, single_coordinator)
        except Exception as e:
            log.error(f"[Thread-{thread_id}] CRASH on {base_name}: {e}")
            success = False

        with lock:
            results["ok" if success else "fail"] = \
                results.get("ok" if success else "fail", 0) + 1

        log.info(f"[Thread-{thread_id}] DONE: {base_name} ({'OK' if success else 'FAILED'})")
        book_queue.task_done()

# Same logic as process_book() but uses a thread-local coordinator instead of the global one, so each thread uses its own API key
def _process_book_with_client(book_dir, base_name, clean_file, chars_file, local_coordinator) -> bool:
    book_status = progress_tracker.book_status(base_name)
    if book_status == "completed":
        return True

    try:
        clean_txt = clean_file.read_text(encoding="utf-8").strip()
        chars_txt = chars_file.read_text(encoding="utf-8").strip()
    except Exception as e:
        log.error(f"  Cannot read: {e}")
        progress_tracker.set_book_status(base_name, "failed", error_msg=str(e))
        return False

    partitions = parse_partitions(clean_txt)
    if not partitions:
        progress_tracker.set_book_status(base_name, "failed", error_msg="no partitions")
        return False

    character_context = parse_characters(chars_txt)
    book_name = base_name.replace("-", " ").replace("_", " ")
    progress_tracker.set_book_status(base_name, "processing", total_parts=len(partitions))

    done_parts = set(progress_tracker.completed_partitions(base_name))
    network_file = book_dir / f"{base_name}_network.csv"
    all_rels = []
    if network_file.exists() and network_file.stat().st_size > 0:
        try:
            existing = pd.read_csv(network_file)
            # Convert back to dicts for history context building
            all_rels = existing.rename(columns={
                "Chapter": "chapter", "Character A": "char_a",
                "Character B": "char_b", "Relationship": "relationship"
            }).to_dict("records")
        except Exception:
            pass

    log_file = book_dir / f"{base_name}_processing_log.csv"
    log_fields = ["partition_num", "partition_title", "n_relationships", "api_key", "timestamp", "status"]
    if not log_file.exists():
        with open(log_file, "w", newline="", encoding="utf-8") as f:
            csv.DictWriter(f, fieldnames=log_fields).writeheader()

    for partition in partitions:
        if partition.number in done_parts:
            continue

        history_ctx = build_history_context(all_rels, partition.number)
        prompt = build_prompt(book_name, character_context, partition, history_ctx)
        messages = [{"role": "user", "content": prompt}]

        # Use this thread's local coordinator
        rels = []
        for attempt in range(MAX_RETRIES):
            try:
                raw, api_name, proc_time, pt, ct = local_coordinator.call(messages)
                raw = re.sub(r'^```(?:json)?\s*', '', raw)
                raw = re.sub(r'\s*```$', '', raw).strip()
                parsed = json.loads(raw)
                if isinstance(parsed, list):
                    rels = [r for r in parsed
                            if all(k in r for k in
                                   ["chapter","char_a","char_b","relationship"])
                            and r["relationship"] in
                            ("positive","negative","neutral")]
                    progress_tracker.log_llm_call(
                        base_name, partition.number, api_name,
                        True, proc_time, pt, ct)
                    break
            except json.JSONDecodeError:
                pass
            except Exception as e:
                log.warning(f"  Retry {attempt+1}: {e}")
                time.sleep(RETRY_DELAY)

        all_rels.extend(rels)

        # Write CSV immediately (crash-safe) — rename columns to match SN format
        if all_rels:
            df_out = pd.DataFrame(all_rels).rename(columns={
                "chapter": "Chapter", "char_a": "Character A",
                "char_b": "Character B", "relationship": "Relationship",
            })
            df_out.to_csv(network_file, index=False, encoding="utf-8")

        with open(log_file, "a", newline="", encoding="utf-8") as f:
            csv.DictWriter(f, fieldnames=log_fields).writerow({
                "partition_num":   partition.number,
                "partition_title": partition.title,
                "n_relationships": len(rels),
                "api_key":         api_name if rels else "unknown",
                "timestamp":       time.strftime("%Y-%m-%d %H:%M:%S"),
                "status":          "completed",
            })

        progress_tracker.mark_partition_done(base_name, partition.number, len(rels))

    progress_tracker.set_book_status(base_name, "completed")
    log.info(f"   {base_name}: {len(all_rels)} relationships")
    return True

# One thread per API key. Books distributed across threads via a shared queue.
def run_threaded():
    books = find_book_directories()
    pending = [(bd, bn, cf, chf) for bd, bn, cf, chf in books
               if progress_tracker.book_status(bn) != "completed"]

    log.info(f"{'='*60}")
    log.info(f"THREADED EXTRACTION — {len(API_CONFIGS)} threads, {len(pending)} books pending")
    log.info(f"{'='*60}")

    book_queue = Queue()
    for item in pending:
        book_queue.put(item)

    results = {"ok": 0, "fail": 0}
    lock = threading.Lock()
    threads = []

    for i, config in enumerate(API_CONFIGS):
        t = threading.Thread(target=worker, args=(i, config, book_queue, results, lock), daemon=True)
        t.start()
        threads.append(t)
        log.info(f"Started thread {i}: {config['name']}")

    # Wait for all books to be processed
    book_queue.join()
    for t in threads:
        t.join(timeout=5)

    log.info(f"\n{'='*60}")
    log.info(f"COMPLETE — OK: {results.get('ok',0)}, Failed: {results.get('fail',0)}")

run_threaded()

In [ ]:
# PROGRESS MONITOR

def show_progress():
    s = progress_tracker.summary()
    books_all = find_book_directories()
    total     = len(books_all)
    done      = s["books"].get("completed", 0)
    failed    = s["books"].get("failed", 0)
    proc      = s["books"].get("processing", 0)
    pending   = total - done - failed - proc
    pct       = done / total * 100 if total else 0

    bar   = "█" * int(pct/2) + "░" * (50 - int(pct/2))
    print(f"\n EXTRACTION PROGRESS")
    print(f"   [{bar}] {pct:.1f}%")
    print(f"   Completed: {done}/{total} | Failed: {failed} | Processing: {proc} | Pending: {pending}")
    print(f"   LLM calls: {s['total_calls']} ({s['success_calls']} OK)")

show_progress()

In [ ]:
# COMBINE ALL PER-BOOK NETWORKS INTO ONE CSV (run after extraction is complete)

# Collects all per-book *_network.csv files under root and concatenates them into a single flat CSV. Adds 'Book' (slug) column for identification.
def combine_networks(root: Path = DATA_ROOT, output_path: Path = OUTPUT_ROOT / "combined_networks.csv") -> pd.DataFrame:
    all_dfs = []
    for book_dir, slug, _, _ in find_book_directories(root):
        net_file = book_dir / f"{slug}_network.csv"
        if net_file.exists() and net_file.stat().st_size > 0:
            df = pd.read_csv(net_file)
            df.insert(0, "Book", slug)
            all_dfs.append(df)

    if not all_dfs:
        print("No network files found. Run the extraction first.")
        return pd.DataFrame()

    combined = pd.concat(all_dfs, ignore_index=True)
    combined.to_csv(output_path, index=False, encoding="utf-8")
    print(f" Combined {len(all_dfs)} books → {len(combined):,} rows")
    print(f" Saved: {output_path}")
    return combined

combined_df = combine_networks()

In [ ]:
# TOKEN USAGE REPORT (run after extraction)

def token_usage_report():
    with progress_tracker._conn() as conn:
        rows = conn.execute('''
            SELECT
                api_key,
                COUNT(*)                    AS calls,
                SUM(success)                AS successful,
                SUM(prompt_tokens)          AS prompt_tokens,
                SUM(completion_tokens)      AS completion_tokens,
                SUM(prompt_tokens + completion_tokens) AS total_tokens,
                ROUND(AVG(proc_time), 2)    AS avg_sec_per_call
            FROM llm_usage
            GROUP BY api_key
        ''').fetchall()

        total = conn.execute('''
            SELECT COUNT(*), SUM(prompt_tokens), SUM(completion_tokens),
                   SUM(prompt_tokens + completion_tokens)
            FROM llm_usage WHERE success=1
        ''').fetchone()

    print("=" * 65)
    print("TOKEN USAGE REPORT — Extraction")
    print("=" * 65)
    print(f"{'API Key':<25} {'Calls':>6} {'OK':>5} {'Prompt':>10} "
          f"{'Completion':>11} {'Total':>10} {'Avg s':>7}")
    print("-" * 65)
    for r in rows:
        print(f"{r[0]:<25} {r[1]:>6} {r[2]:>5} {r[3]:>10,} "
              f"{r[4]:>11,} {r[5]:>10,} {r[6]:>7}")
    print("=" * 65)
    print(f"{'TOTAL':<25} {total[0]:>6}      {total[1]:>10,} "
          f"{total[2]:>11,} {total[3]:>10,}")
    print()
    # Rough cost estimate for DeepInfra (~$0.13/1M tokens for Llama 70B)
    cost = (total[3] or 0) / 1_000_000 * 0.13
    print(f"Estimated cost (DeepInfra ~$0.13/1M tokens): ${cost:.4f}")
    print(f"Note: AcademicCloud usage is free/quota-based.")

token_usage_report()